# FormulaLM — PROMPT 10: Merge `v2-ship`, Export GGUFs, Verify Prompt Parity & Quant Eval

Runs on **Google Colab or Kaggle T4 GPU** (keeps Abubakar's 8 GB CPU-only laptop free of heavy merging/conversion/full-split eval):

1. **Merge (`v2-ship`):** loads pinned `Qwen/Qwen3-1.7B@70d244cc86ccca08cf5af4e1e306ecf908b1ad5e` in fp16 (explicitly overriding the 4-bit mirror in `adapter_config.json`), applies `abubakar56/formulalm-qwen3-1.7b-lora-v2@e2dd7e8115406ffcdbbdfa60d6f2f10b4aaf2595`, merges to 16-bit, persists non-thinking chat template, and checks byte-for-byte tokenizer parity.
2. **Convert & Quantize (`llama.cpp`):** exports `F16` (reference), `Q8_0`, `Q5_K_M`, and `Q4_K_M` GGUFs and logs exact sizes + SHA-256 hashes in `results/quant_artifacts.json`.
3. **20-Example Prompt Parity Check:** compares merged HF FP16 outputs against GGUF F16 (and Q4_K_M) on 20 fixed examples from `test_real_v2.csv`, reporting any difference in `results/quant_parity/parity_report.md`.
4. **Full Quality Eval (`results/quant_*/`):** evaluates `F16`, `Q8_0`, `Q5_K_M`, and `Q4_K_M` on `val`, `test_real`, `test_real_v2`, and `test_synthetic` via `run_eval.py` (`LlamaCppBackend`, identical `seed=42`, `temperature=0`, `max_tokens=96`, `few_shot=0`, `enable_thinking=False`).
5. **Comparison Table:** generates `results/quant_comparison.md` and pushes lightweight `results/` back to GitHub.
6. **HF Upload (GATED):** `PUSH_GGUF_TO_HF = False` by default — push GGUFs to Hugging Face ONLY after Abubakar reviews parity & quality results and confirms.

In [ ]:
# 1. Check GPU & optionally mount Google Drive to persist exported GGUFs
import pathlib, torch

HAS_CUDA = torch.cuda.is_available()
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print('WARNING: No CUDA GPU detected. Merging/conversion works on CPU, but Cell 6 full eval will be slow.')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

In [ ]:
# 2. Clone / pull FormulaLM (Colab or Kaggle), install deps, ensure val.jsonl & test_synthetic.jsonl exist
import getpass, os, pathlib, subprocess, sys

IS_KAGGLE = pathlib.Path('/kaggle/working').is_dir()
REPO_DIR = pathlib.Path('/kaggle/working/FormulaLM' if IS_KAGGLE else '/content/FormulaLM')

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _ks = UserSecretsClient()
        for _key in ('GITHUB_TOKEN', 'HF_TOKEN'):
            try:
                os.environ[_key] = _ks.get_secret(_key)
                print(f'Loaded Kaggle secret: {_key}')
            except Exception:
                pass
    except Exception as exc:
        print('kaggle_secrets unavailable:', exc)

if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
for _p in (str(REPO_DIR), str(REPO_DIR / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.environ['PYTHONPATH'] = f"{REPO_DIR}:{REPO_DIR / 'src'}"

%pip uninstall -y -q torchao
%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft sentencepiece protobuf gguf "transformers>=4.51.0" "huggingface_hub>=0.30.0"
%pip install -q --no-deps -e .

if not pathlib.Path('data/processed/val.jsonl').is_file() or not pathlib.Path('data/processed/test_synthetic.jsonl').is_file():
    !python -m src.data.build_dataset --version v2 --workers 2

In [ ]:
# 3. Merge v2-ship LoRA adapter onto pinned Qwen/Qwen3-1.7B@70d244cc (fp16) + capture 20 HF parity outputs
import gc, json, pathlib, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from scripts.train_qlora import configure_qwen3_non_thinking
from src.eval.prompt_parity import run_parity_capture, verify_tokenizer_parity
from src.eval.run_eval import TransformersBackend

BASE_MODEL_ID = 'Qwen/Qwen3-1.7B'
BASE_REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
ADAPTER_REPO = 'abubakar56/formulalm-qwen3-1.7b-lora-v2'
ADAPTER_REVISION = 'e2dd7e8115406ffcdbbdfa60d6f2f10b4aaf2595'

WORK_DIR = pathlib.Path('/kaggle/working/formulalm_quant' if pathlib.Path('/kaggle/working').is_dir() else '/content/formulalm_quant')
MERGED_DIR = WORK_DIR / 'merged_fp16'
GGUF_DIR = WORK_DIR / 'gguf'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

print(f'Loading base {BASE_MODEL_ID}@{BASE_REVISION[:8]} in float16...', flush=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION)
configure_qwen3_non_thinking(tokenizer)
tok_check = verify_tokenizer_parity(tokenizer)
assert tok_check['exact_match'], f'Chat template parity mismatch: {tok_check}'
print('Tokenizer non-thinking template parity: EXACT MATCH', flush=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_REVISION,
    torch_dtype=torch.float16,
    device_map='auto' if torch.cuda.is_available() else 'cpu',
)
print(f'Applying adapter {ADAPTER_REPO}@{ADAPTER_REVISION[:8]}...', flush=True)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO, revision=ADAPTER_REVISION)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print('Saved merged 16-bit HF model to:', MERGED_DIR, flush=True)

del merged_model, peft_model, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Capture 20 fixed parity outputs directly from the saved merged HF FP16 model
hf_merged_backend = TransformersBackend(str(MERGED_DIR), load_in_4bit=False, enable_thinking=False)
hf_parity = run_parity_capture(
    hf_merged_backend,
    dataset_path='data/test_real_v2/test_real_v2.csv',
    n=20,
    seed=42,
    max_tokens=96,
    out_jsonl='results/quant_parity/hf_merged_fp16_20.jsonl',
)
print(f"Captured 20 HF Merged FP16 parity outputs: {sum(r['exec_correct'] for r in hf_parity)}/20 exec correct", flush=True)
del hf_merged_backend
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 4. Build llama.cpp (llama-quantize + llama-server) and export F16, Q8_0, Q5_K_M, Q4_K_M GGUFs
import hashlib, json, pathlib, shutil, subprocess, torch

LLAMA_DIR = WORK_DIR / 'llama.cpp'
if not LLAMA_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)], check=True)

llama_commit = subprocess.check_output(['git', '-C', str(LLAMA_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('llama.cpp commit:', llama_commit, flush=True)

BUILD_DIR = LLAMA_DIR / 'build'
cmake_args = ['cmake', '-B', str(BUILD_DIR), '-S', str(LLAMA_DIR), '-DCMAKE_BUILD_TYPE=Release', '-DGGML_CUDA=' + ('ON' if torch.cuda.is_available() else 'OFF')]
subprocess.run(cmake_args, check=True)
subprocess.run(['cmake', '--build', str(BUILD_DIR), '--target', 'llama-quantize', 'llama-server', '-j'], check=True)

QUANTIZE_BIN = BUILD_DIR / 'bin' / 'llama-quantize'
SERVER_BIN = BUILD_DIR / 'bin' / 'llama-server'
assert QUANTIZE_BIN.is_file() and SERVER_BIN.is_file(), 'llama.cpp binaries missing!'

F16_GGUF = GGUF_DIR / 'formulalm-qwen3-1.7b-v2-f16.gguf'
if not F16_GGUF.is_file():
    subprocess.run(['python', str(LLAMA_DIR / 'convert_hf_to_gguf.py'), str(MERGED_DIR), '--outfile', str(F16_GGUF), '--outtype', 'f16'], check=True)

QUANTS = {
    'f16': ('F16', F16_GGUF),
    'q8_0': ('Q8_0', GGUF_DIR / 'formulalm-qwen3-1.7b-v2-q8_0.gguf'),
    'q5_k_m': ('Q5_K_M', GGUF_DIR / 'formulalm-qwen3-1.7b-v2-q5_k_m.gguf'),
    'q4_k_m': ('Q4_K_M', GGUF_DIR / 'formulalm-qwen3-1.7b-v2-q4_k_m.gguf'),
}

for key, (qtype, out_path) in QUANTS.items():
    if key == 'f16':
        continue
    if not out_path.is_file():
        print(f'Quantizing F16 -> {qtype} ({out_path.name})...', flush=True)
        subprocess.run([str(QUANTIZE_BIN), str(F16_GGUF), str(out_path), qtype], check=True)

def sha256_file(p: pathlib.Path) -> str:
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

artifacts = {
    'base_model_id': BASE_MODEL_ID,
    'base_revision': BASE_REVISION,
    'adapter_repo': ADAPTER_REPO,
    'adapter_revision': ADAPTER_REVISION,
    'llama_cpp_commit': llama_commit,
    'files': {},
}
for key, (qtype, p) in QUANTS.items():
    sz = p.stat().st_size
    artifacts['files'][key] = {
        'quant_type': qtype,
        'filename': p.name,
        'size_bytes': sz,
        'size_mb': round(sz / (1024 * 1024), 2),
        'sha256': sha256_file(p),
    }
    print(f"{qtype:6s} | {p.name:36s} | {sz/(1024*1024):7.2f} MB | sha256={artifacts['files'][key]['sha256'][:16]}...")

pathlib.Path('results/quant_artifacts.json').write_text(json.dumps(artifacts, indent=2) + '\n', encoding='utf-8')

# Copy exported GGUFs + Modelfiles to Google Drive if mounted so you can download Q4_K_M / Q5_K_M / Q8_0 to your laptop
DRIVE_GGUF = pathlib.Path('/content/drive/MyDrive/FormulaLM/gguf_v2')
if pathlib.Path('/content/drive/MyDrive').is_dir():
    DRIVE_GGUF.mkdir(parents=True, exist_ok=True)
    for _, (_, p) in QUANTS.items():
        dst = DRIVE_GGUF / p.name
        if not dst.is_file() or dst.stat().st_size != p.stat().st_size:
            shutil.copy2(p, dst)
    for mf in pathlib.Path('deploy').glob('Modelfile*'):
        shutil.copy2(mf, DRIVE_GGUF / mf.name)
    print('Copied GGUFs + Modelfiles to:', DRIVE_GGUF)

In [ ]:
# 5. Verify prompt-format & output parity on 20 fixed examples (HF Merged FP16 vs GGUF F16 and Q4_K_M)
import json, pathlib, requests, subprocess, time, torch
from src.eval.prompt_parity import compare_parity_records, load_parity_records, render_parity_markdown, run_parity_capture
from src.eval.run_eval import LlamaCppBackend

def start_llama_server(gguf_path: pathlib.Path, port: int = 8080) -> subprocess.Popen:
    ngl = '99' if torch.cuda.is_available() else '0'
    proc = subprocess.Popen(
        [str(SERVER_BIN), '-m', str(gguf_path), '-ngl', ngl, '-c', '2048', '--port', str(port), '--host', '127.0.0.1'],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
    )
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError(f'llama-server exited early: {proc.stderr.read() if proc.stderr else ""}')
        try:
            if requests.get(f'http://127.0.0.1:{port}/health', timeout=2).status_code == 200:
                return proc
        except Exception:
            pass
        time.sleep(1)
    proc.terminate()
    raise TimeoutError(f'llama-server did not become healthy for {gguf_path.name}')

hf_records = load_parity_records('results/quant_parity/hf_merged_fp16_20.jsonl')
for qkey, label in [('f16', 'GGUF F16'), ('q4_k_m', 'GGUF Q4_K_M')]:
    qtype, gguf_path = QUANTS[qkey]
    srv = start_llama_server(gguf_path, port=8080)
    try:
        backend = LlamaCppBackend(
            base_url='http://127.0.0.1:8080',
            model=gguf_path.name,
            use_raw_prompt=True,
            base_model_id=BASE_MODEL_ID,
            base_revision=BASE_REVISION,
            quantization=qtype,
        )
        cand_records = run_parity_capture(
            backend,
            dataset_path='data/test_real_v2/test_real_v2.csv',
            n=20,
            seed=42,
            max_tokens=96,
            out_jsonl=f'results/quant_parity/gguf_{qkey}_20.jsonl',
        )
    finally:
        srv.terminate()
        srv.wait(timeout=15)

    rep = compare_parity_records(hf_records, cand_records, ref_label='HF Merged FP16', cand_label=label)
    md = render_parity_markdown(rep)
    suffix = '' if qkey == 'f16' else f'_{qkey}'
    pathlib.Path(f'results/quant_parity/parity_report{suffix}.md').write_text(md, encoding='utf-8')
    pathlib.Path(f'results/quant_parity/parity_report{suffix}.json').write_text(json.dumps(rep, indent=2) + '\n', encoding='utf-8')
    print(md, flush=True)

In [ ]:
# 6. Full quality evaluation for F16, Q8_0, Q5_K_M, Q4_K_M on val, test_real, test_real_v2, test_synthetic
from src.eval.run_eval import LlamaCppBackend, evaluate

EVAL_SPLITS = [
    ('val', 'data/processed/val.jsonl'),
    ('real_v2', 'data/test_real_v2/test_real_v2.csv'),
    ('real', 'data/test_real/test_real_DRAFT.csv'),
    ('synthetic', 'data/processed/test_synthetic.jsonl'),
]

for qkey in ('f16', 'q8_0', 'q5_k_m', 'q4_k_m'):
    qtype, gguf_path = QUANTS[qkey]
    print(f'\n========== Evaluating {qtype} ({gguf_path.name}) ==========', flush=True)
    srv = start_llama_server(gguf_path, port=8080)
    try:
        backend = LlamaCppBackend(
            base_url='http://127.0.0.1:8080',
            model=gguf_path.name,
            use_raw_prompt=True,
            base_model_id=BASE_MODEL_ID,
            base_revision=BASE_REVISION,
            quantization=qtype,
        )
        for split_key, ds_path in EVAL_SPLITS:
            run_name = f'quant_{qkey}_{split_key}'
            print(f'--- Running {run_name} ({ds_path}) ---', flush=True)
            m = evaluate(
                backend=backend,
                dataset_path=ds_path,
                run_name=run_name,
                seed=42,
                max_tokens=96,
                few_shot=0,
                batch_size=1,
                resume=True,
            )
            ov = m['overall']
            print(
                f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
                f"exec_acc={100*ov['execution_accuracy']:.1f}% | exact={100*ov['exact_match_rate']:.1f}% | "
                f"avg_lat={ov['avg_latency_ms']:.1f}ms",
                flush=True,
            )
    finally:
        srv.terminate()
        srv.wait(timeout=15)

In [ ]:
# 7. Generate results/quant_comparison.md and push lightweight results back to GitHub (NO GGUFs committed!)
!python -m src.eval.compare_quant \
  --results-dir results \
  --out results/quant_comparison.md \
  --json-out results/quant_comparison.json

PUSH_RESULTS_TO_GITHUB = True
if PUSH_RESULTS_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/quant_* results/quant_artifacts.json results/quant_parity results/quant_comparison.md results/quant_comparison.json
    !git diff --cached --quiet || git commit -m "results: add v2-ship GGUF parity check and F16/Q8_0/Q5_K_M/Q4_K_M eval results"
    !git pull --rebase origin main
    !git push origin main

In [ ]:
# 8. GATED — Push GGUFs to Hugging Face ONLY AFTER Abubakar reviews parity & quant_comparison.md and confirms
PUSH_GGUF_TO_HF = False  # <-- Leave False until you confirm!
HF_GGUF_REPO = 'abubakar56/formulalm-qwen3-1.7b-gguf'

if PUSH_GGUF_TO_HF:
    import getpass, os
    from huggingface_hub import HfApi
    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            hf_token = getpass.getpass('Hugging Face write token (hidden): ').strip()
    api = HfApi(token=hf_token)
    api.create_repo(HF_GGUF_REPO, exist_ok=True, repo_type='model')
    for qkey in ('q4_k_m', 'q5_k_m', 'q8_0', 'f16'):
        _, p = QUANTS[qkey]
        print(f'Uploading {p.name}...', flush=True)
        api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name, repo_id=HF_GGUF_REPO, repo_type='model')
    for mf in pathlib.Path('deploy').glob('Modelfile*'):
        api.upload_file(path_or_fileobj=str(mf), path_in_repo=mf.name, repo_id=HF_GGUF_REPO, repo_type='model')
    print('Uploaded GGUFs + Modelfiles to https://huggingface.co/' + HF_GGUF_REPO)
else:
    print('HF upload skipped (PUSH_GGUF_TO_HF=False). Confirm parity & quant_comparison.md first.')